# Day 26 — Tool-Using Agents with Chainlit UI

**Intern:** Sehar Andleeb  
**Company:** Xeven Solutions  
**Date:** June 21, 2026

---

Today I built a conversational AI agent that uses four tools to answer questions it can't answer from memory alone. The agent runs inside a Chainlit chat interface (dark theme, file upload, conversation memory) and routes each question to the right tool automatically.

**Tools built:**
- Calculator — safe math expression evaluator
- Web Search — live DuckDuckGo results
- RAG — hybrid search over an arXiv paper (Day 25 pipeline)
- Weather — live weather via Open-Meteo API


## What I Learned

### ReAct vs. Tool-Calling Agents

I started with `create_react_agent`, which prompts the LLM to produce text in a `Thought / Action / Observation` format that gets parsed back into tool calls. This kept hitting `max_iterations` because the small model sometimes produced slightly wrong formatting, causing parse failures and loops.

I switched to `create_tool_calling_agent`, which uses the model's native function-calling API. Instead of parsing text, the model emits structured tool calls directly. This is faster, more reliable, and needs no text parsing.

### Safe Calculator

A plain `eval()` is a code-execution hole — any input that reaches it can run arbitrary Python. I used Python's `ast` module instead: parse the expression into a tree, walk the tree, and only allow numeric literals and arithmetic operators. A `__import__('os').system(...)` injection gets rejected at the parse step, before any execution happens.

### Prompt Engineering

The system prompt went through several iterations:
1. First version — agent over-used tools, looping on questions it already knew
2. Added "answer from your own knowledge when confident" — agent then hallucinated weather instead of calling the tool
3. Added explicit tool-to-use-case mapping — agent leaked internal reasoning into final answers
4. Final version — short, unambiguous, one rule per tool, no vague heuristics

The lesson: a shorter, more specific prompt beats a longer list of general rules.

### Groq Quota Management

The Groq free tier has two limits: tokens per minute (TPM) and tokens per day (TPD). Day 25's RAG reranker was hardcoded to `llama-3.3-70b-versatile`, which burns the daily budget much faster than `llama-3.1-8b-instant`. One line change in `reranker.py` fixed quota exhaustion, reduced latency, and made RAG reliable again.

### Conversation Memory

Each Chainlit session stores the last 6 exchanges as `HumanMessage` / `AIMessage` objects, passed back into the agent on every turn as `chat_history`. Follow-up questions like "explain that simpler" work correctly because the agent sees the previous answer.

### File Upload

Uploaded PDFs are extracted with PyMuPDF, split into 800-character chunks, and for each question the most relevant chunks are selected by keyword overlap. Full-document injection was tried first but immediately exceeded Groq's TPM limit (a 74,500-character PDF is ~18,000 tokens, well above the 6,000 TPM cap). Keyword selection keeps each request under ~3,000 tokens.


## Design Decisions

| Question | What I tried first | What I used and why |
|----------|-------------------|---------------------|
| ReAct vs tool-calling? | `create_react_agent` | `create_tool_calling_agent` — no text parsing, fewer failures |
| Safe math eval? | `eval()` | `ast` tree walk — rejects injection at parse time |
| Web search? | Paid APIs | `ddgs` (DuckDuckGo) — free, no API key |
| Live weather? | OpenWeatherMap | Open-Meteo — free, no API key, geocoding built in |
| Chat UI? | Custom FastAPI + vanilla JS | Chainlit — production UX out of the box, per mentor request |
| Document retrieval? | Full-document injection | Keyword-overlap chunk selection — stays within TPM limit |
| Which Groq model? | `llama-3.3-70b-versatile` | `llama-3.1-8b-instant` — faster, quota-efficient |


## Setup

In [11]:
# Packages used today
# uv pip install langchain langchain-groq ddgs chainlit pymupdf requests

# Confirm tools folder is importable
import sys
sys.path.insert(0, ".")
print("Path set up, importing tools...")

Path set up, importing tools...


## Tool 1 — Calculator (`tools/calculator_tool.py`)

Parses math expressions with Python's `ast` module. Only allows numeric literals and arithmetic operators. Anything else is rejected.

![Calculator tool in action](screenshots/calculator-demo.png)


In [12]:
from tools.calculator_tool import calculator

# Normal math
print(calculator.invoke("23 * 45"))
print(calculator.invoke("10 / 0"))

# Code injection attempt — safely rejected
print(calculator.invoke("__import__('os').system('echo hacked')"))

1035
Error: division by zero.
Error: could not evaluate '__import__('os').system('echo hacked')' (Unsupported expression node: Call(func=Attribute(value=Call(func=Name(id='__import__', ctx=Load()), args=[Constant(value='os')], keywords=[]), attr='system', ctx=Load()), args=[Constant(value='echo hacked')], keywords=[])).


## Tool 2 — Web Search (`tools/web_search_tool.py`)

Uses `ddgs` (DuckDuckGo Search) for live results with no API key. Returns top results with title, snippet, and source URL.

> **Note:** web-search screenshot to be added — Groq daily quota (97,600/100,000 tokens) was exhausted during testing. The tool itself works correctly as shown in the terminal output below.


In [13]:
from tools.web_search_tool import web_search

results = web_search.invoke("LangChain agents 2026")
print(results)

Top web results for 'LangChain agents 2026':
1. LangChain Agents in 2026: The Complete Guide... — EasyClaw
   Final Verdict — LangChain Agents in 2026: Still Worth It? Yes — with one major caveat. LangChain's ecosystem advantage is real.
   Source: https://easyclaw.com/blog/ai-agent-101/langchain-agents-complete-guide/
2. LangChain Agents Tools Guide 2026: Functions Explained
   Master langchain agents tools and functions with this practical 2026 guide. Learn how to build reasoning agents, wire up custom tools, and ship production AI apps.
   Source: https://freeacademy.ai/blog/langchain-agents-tools-functions-practical-guide-2026
3. LangChain Agents Deep Dive: The Ultimate Guide... - DEV Community
   This article systematically dissects the architecture of LangChain Agents, core concepts, practical patterns, and best practices within the 2026 technical ecosystem.
   Source: https://dev.to/jearick/langchain-agents-deep-dive-the-ultimate-guide-to-building-intelligent-agents-in-2026-4b8p

## Tool 3 — RAG (`tools/rag_tool.py`)

Wraps Day 25's `RagService`: hybrid FAISS + BM25 retrieval over the ingested "Attention Is All You Need" paper, with LLM reranking. The reranker model was changed from `llama-3.3-70b-versatile` to `llama-3.1-8b-instant` to fix daily quota exhaustion.

![RAG tool — self-attention answer grounded in the paper](screenshots/rag-demo.png)


In [14]:
from tools.rag_tool import rag_search

results = rag_search.invoke("What is self-attention?")
print(results)

Top document excerpts for 'What is self-attention?':
1. [Attention Is All You Need - 4 Why Self-Attention] (score=20.000)
   the path length between long-range dependencies in the network. Learning long-range dependencies is a key challenge in many sequence transduction tasks. One key factor affecting the ability to learn such dependencies is the length of the paths forward and backward signals have to traverse in the net...
2. [Attention Is All You Need - 4 Why Self-Attention] (score=19.000)
   faster than recurrent layers when the sequence length n 𝑛 n is smaller than the representation dimensionality d 𝑑 d , which is most often the case with sentence representations used by state-of-the-art models in machine translations, such as word-piece [ 38 ] and byte-pair [ 31 ] representations. To...
3. [Attention Is All You Need - 3 Model Architecture] (score=18.000)
   Where the projections are parameter matrices W i Q ∈ ℝ d model × d k subscript superscript 𝑊 𝑄 𝑖 superscript ℝ subscript 𝑑 

## Tool 4 — Weather (`tools/weather_tool.py`)

Uses Open-Meteo's free API. Geocodes the city name, fetches current temperature, humidity, wind speed, and weather code, and maps the weather code to a human-readable description. No API key required.

![Weather tool — live weather for Lahore](screenshots/weather-demo.png)


In [15]:
from tools.weather_tool import weather

print(weather.invoke("Lahore"))
print(weather.invoke("Tokyo"))

Current weather in Lahore, Pakistan: 33.6°C, overcast, humidity 41%, wind 3.3 km/h.
Current weather in Tokyo, Japan: 25.9°C, overcast, humidity 66%, wind 3.5 km/h.


## Full Agent — All Four Tools Wired Together

`create_tool_calling_agent` + `AgentExecutor` handles routing. The model decides which tool to call (or none) based on the question, using native function-calling.

![Chainlit UI — empty state](screenshots/chainlit-ui.png)


In [16]:
from agent import build_agent

executor = build_agent()

questions = [
    "What is AI?",
    "What is 23 * 45?",
    "What is self-attention according to the paper?",
    "What is the weather in Lahore right now?",
]

for q in questions:
    result = executor.invoke({"input": q, "chat_history": []})
    print(f"Q: {q}")
    print(f"A: {result['output']}")
    print()

Q: What is AI?
A: AI, or Artificial Intelligence, refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. The term can also be applied to any machine that exhibits traits associated with a human mind such as learning and problem-solving.

Q: What is 23 * 45?
A: The result of 23 * 45 is 1035.



APIError: Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.

## Conversation Memory

The agent stores the last 6 exchanges per session and passes them back as `chat_history` on every turn. The screenshot below shows a follow-up question ("explain it") correctly elaborating on the previous answer without the user repeating context.

![Conversation memory — follow-up question works correctly](screenshots/conversation-memory-demo.png)


## File Upload — PDF Question Answering

Users can attach a PDF mid-conversation. The file is extracted with PyMuPDF, split into 800-character chunks, and the most relevant chunks are selected by keyword overlap for each question. This keeps requests within Groq's per-minute token limit while giving the agent relevant content.

![File upload — PDF ingested and ready for questions](screenshots/file-upload-demo.png)


## Architecture

```
User message (Chainlit)
        │
        ▼
chainlit_app.py
  ├── extract file chunks (PyMuPDF) if PDF uploaded
  ├── select relevant chunks by keyword overlap
  ├── append chat_history (last 6 turns)
        │
        ▼
AgentExecutor
  ├── create_tool_calling_agent
  ├── ChatGroq — llama-3.1-8b-instant (temperature=0)
  └── Tools:
       ├── calculator      → ast-based safe eval
       ├── web_search_tool → ddgs DuckDuckGo
       ├── rag_tool        → Day 25 RagService (FAISS+BM25+rerank)
       └── weather_tool    → Open-Meteo geocode + forecast
        │
        ▼
Answer → Chainlit message bubble
```

**Run the Chainlit UI:**
```
cd day26
chainlit run chainlit_app.py -w
```
Opens at `http://localhost:8000`

## What I Would Improve Next

- Replace keyword-overlap chunk selection with embedding-based retrieval for uploaded documents
- Stream intermediate tool-call steps to the UI so users can see what the agent is doing while it thinks
- Persist conversation history to disk so sessions survive server restarts
- Add a result cache for web search to avoid repeated DDG calls on similar queries
